<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/05_NeuroFHIR_QC_Trust_and_Robustness_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/05_NeuroFHIR_QC_Trust_and_Robustness_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeuroFHIR-QC — Notebook 05
## Trust, Perturbation Robustness, QC Triage, and Blocked Finalization

**Runtime:** Google Colab GPU; L4 recommended  
**Inputs:** completed Notebook 04 segmentation artifacts  
**Data boundary:** public de-identified MRI linked only to synthetic FHIR demonstration context

This notebook runs four controlled perturbations for each of the three cases, adds one explicitly synthetic severe challenge for the locked low-confidence case, calculates segmentation stability and plausibility indicators, assigns an engineering QC category, and blocks autonomous finalization.

The categories are **High confidence**, **Review recommended**, and **Manual review required**. They are workflow-design signals—not clinically validated safety probabilities. Every AI result remains `preliminary` until explicit human review.

Notebook 05 does not calculate longitudinal change, create a current FHIR AI Observation, execute a reviewer decision, or perform FHIR write-back.

In [1]:
# Cell 1 — Mount Drive, load project state, and enforce the Notebook 04 completion gate

from __future__ import annotations

import csv
import hashlib
import importlib.util
import json
import re
import shutil
import subprocess
import sys
import textwrap
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

from google.colab import drive
drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")
CONFIG_PATH = PROJECT_ROOT / "project_config.json"
NOTEBOOK_MANIFEST_PATH = PROJECT_ROOT / "notebook_manifest.json"
NB04_AUDIT_PATH = PROJECT_ROOT / "evaluation/results/notebook_04_segmentation_volumetry_audit.json"
SEGMENTATION_MANIFEST_PATH = PROJECT_ROOT / "data/sample_masks/notebook_04/segmentation_case_manifest.json"
SEGMENTATION_METRICS_PATH = PROJECT_ROOT / "evaluation/results/notebook_04_segmentation_and_volumetry/segmentation_metrics.json"
IMAGING_MANIFEST_PATH = PROJECT_ROOT / "data/sample_images/notebook_03/imaging_case_manifest.json"
SEGMENTATION_SERVICE_PATH = PROJECT_ROOT / "backend/app/services/segmentation_service.py"

def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False, allow_nan=False)
        handle.write("\n")
    tmp.replace(path)

def utc_now() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def notebook_entries(manifest: Any) -> list[dict[str, Any]]:
    if isinstance(manifest, list):
        return manifest
    if isinstance(manifest, dict):
        for key in ("notebooks", "entries", "workflow"):
            if isinstance(manifest.get(key), list):
                return manifest[key]
    raise ValueError("Unrecognized notebook_manifest.json structure")

def number_of(value: Any) -> str:
    match = re.search(r"\d+", str(value))
    return match.group(0).zfill(2) if match else str(value)

def find_entry(manifest: Any, number: str) -> dict[str, Any]:
    target = number.zfill(2)
    for entry in notebook_entries(manifest):
        candidates = [entry.get("number"), entry.get("notebook_number"), entry.get("id"), entry.get("filename")]
        if any(number_of(v) == target for v in candidates if v is not None):
            return entry
    raise KeyError(f"Notebook {target} missing from manifest")

required = [
    CONFIG_PATH, NOTEBOOK_MANIFEST_PATH, NB04_AUDIT_PATH,
    SEGMENTATION_MANIFEST_PATH, SEGMENTATION_METRICS_PATH,
    IMAGING_MANIFEST_PATH, SEGMENTATION_SERVICE_PATH,
]
missing = [str(p) for p in required if not p.exists() or p.stat().st_size == 0]
if missing:
    raise FileNotFoundError("Notebook 04 evidence is incomplete:\n" + "\n".join(f" - {p}" for p in missing))

project_config = load_json(CONFIG_PATH)
notebook_manifest = load_json(NOTEBOOK_MANIFEST_PATH)
nb04_audit = load_json(NB04_AUDIT_PATH)
segmentation_manifest = load_json(SEGMENTATION_MANIFEST_PATH)
segmentation_metrics = load_json(SEGMENTATION_METRICS_PATH)
imaging_manifest = load_json(IMAGING_MANIFEST_PATH)

nb04_entry = find_entry(notebook_manifest, "04")
nb05_entry = find_entry(notebook_manifest, "05")
nb04_status = str(nb04_entry.get("status", nb04_audit.get("status", ""))).lower()
if nb04_status not in {"completed", "complete", "passed"}:
    raise RuntimeError(f"Notebook 04 is not complete: {nb04_status!r}")

required_metrics = {
    "case_count": 3,
    "inference_success_rate": 1.0,
    "model_output_nonempty_rate": 1.0,
    "output_shape_integrity_rate": 1.0,
    "output_affine_integrity_rate": 1.0,
    "preview_count": 3,
}
metrics = nb04_audit.get("metrics", {})
failed = [k for k, v in required_metrics.items() if float(metrics.get(k, -1)) != float(v)]
if failed:
    raise RuntimeError("Notebook 04 metrics failed: " + ", ".join(failed))

case_ids = [c["case_id"] for c in segmentation_manifest.get("cases", [])]
if set(case_ids) != {"stable", "progression", "low-confidence"}:
    raise AssertionError(f"Unexpected case ids: {case_ids}")

NOTEBOOK_FILENAME = nb05_entry.get("filename", "05_NeuroFHIR_QC_Trust_and_Robustness_Engine.ipynb")
NOTEBOOK_SAVE_PATH = PROJECT_ROOT / "notebooks" / NOTEBOOK_FILENAME

MASK_ROOT = PROJECT_ROOT / "data/sample_masks/notebook_05"
QC_DATA_ROOT = PROJECT_ROOT / "data/sample_biomarkers/notebook_05"
EVAL_ROOT = PROJECT_ROOT / "evaluation/results/notebook_05_trust_and_robustness"
CASE_RESULT_ROOT = EVAL_ROOT / "case_results"
PREVIEW_ROOT = EVAL_ROOT / "previews"

PERTURBATION_DEFINITIONS_PATH = EVAL_ROOT / "perturbation_definitions.json"
PERTURBATION_RESULTS_JSON = EVAL_ROOT / "perturbation_results.json"
PERTURBATION_RESULTS_CSV = EVAL_ROOT / "perturbation_results.csv"
QC_MANIFEST_PATH = QC_DATA_ROOT / "qc_case_manifest.json"
QC_SUMMARY_JSON = EVAL_ROOT / "qc_case_summary.json"
QC_SUMMARY_CSV = EVAL_ROOT / "qc_case_summary.csv"
RUNTIME_LOG_PATH = EVAL_ROOT / "robustness_runtime_log.json"
AUDIT_JSON_PATH = PROJECT_ROOT / "evaluation/results/notebook_05_trust_robustness_audit.json"
AUDIT_MD_PATH = PROJECT_ROOT / "docs/NOTEBOOK_05_TRUST_AND_ROBUSTNESS_ENGINE.md"

for folder in (MASK_ROOT, QC_DATA_ROOT, EVAL_ROOT, CASE_RESULT_ROOT, PREVIEW_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("✅ Notebook 04 completion gate passed")
print(f"✅ Cases: {sorted(case_ids)}")
print(f"📓 Notebook 05 path: {NOTEBOOK_SAVE_PATH}")
print("⚠️ QC thresholds are engineering workflow parameters, not clinical validation")
print("=" * 100)

Mounted at /content/drive
✅ Notebook 04 completion gate passed
✅ Cases: ['low-confidence', 'progression', 'stable']
📓 Notebook 05 path: /content/drive/MyDrive/neurofhir-qc/notebooks/05_NeuroFHIR_QC_Trust_and_Robustness_Engine.ipynb
⚠️ QC thresholds are engineering workflow parameters, not clinical validation


In [2]:
# Cell 2 — Install runtime packages and load the pinned Notebook 04 model service

packages = [
    "monai==1.6.0",
    "scipy>=1.11,<2",
    "nibabel>=5.2,<6",
    "pandas>=2,<3",
    "matplotlib>=3.8,<4",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])

import matplotlib
import matplotlib.pyplot as plt
import monai
import nibabel as nib
import numpy as np
import pandas as pd
import scipy
import torch
from scipy import ndimage

if not torch.cuda.is_available():
    raise RuntimeError("Choose Runtime → Change runtime type → L4 GPU, restart, and run from Cell 1")

DEVICE = torch.device("cuda:0")
GPU = torch.cuda.get_device_properties(0)
GPU_MEMORY_GIB = GPU.total_memory / (1024 ** 3)

first_case = segmentation_manifest["cases"][0]
ROI_SIZE = tuple(int(v) for v in first_case["roi_size"])
SW_OVERLAP = float(first_case["sliding_window_overlap"])
MODEL_THRESHOLD = float(first_case["threshold"])
USE_AMP = bool(first_case["amp_enabled"])

spec = importlib.util.spec_from_file_location("nqc_segmentation_service", SEGMENTATION_SERVICE_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not import {SEGMENTATION_SERVICE_PATH}")
segmentation_service = importlib.util.module_from_spec(spec)
spec.loader.exec_module(segmentation_service)

model_info = segmentation_manifest.get("model", {})
checkpoint_rel = model_info.get(
    "checkpoint_relative_path",
    "model/brats_mri_segmentation_v0.5.4/models/model.pt",
)
CHECKPOINT_PATH = PROJECT_ROOT / checkpoint_rel
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(CHECKPOINT_PATH)

expected_sha = first_case.get("model_checkpoint_sha256") or model_info.get("checkpoint_sha256")
actual_sha = sha256_file(CHECKPOINT_PATH)
if expected_sha and actual_sha != expected_sha:
    raise AssertionError(f"Checkpoint checksum mismatch: {actual_sha}")

MODEL = segmentation_service.load_brats_model(CHECKPOINT_PATH, DEVICE)
runtime_versions = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "monai": monai.__version__,
    "nibabel": nib.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "matplotlib": matplotlib.__version__,
}

print("=" * 100)
for name, version in runtime_versions.items():
    print(f"{name}: {version}")
print(f"GPU: {GPU.name}")
print(f"GPU memory: {GPU_MEMORY_GIB:.2f} GiB")
print(f"ROI: {ROI_SIZE}; overlap={SW_OVERLAP}; threshold={MODEL_THRESHOLD}")
print("✅ Pinned model loaded and checksum verified")
print("=" * 100)

python: 3.12.13
torch: 2.11.0+cu128
monai: 1.6.0
nibabel: 5.4.2
numpy: 2.0.2
pandas: 2.2.2
scipy: 1.16.3
matplotlib: 3.10.0
GPU: NVIDIA L4
GPU memory: 22.03 GiB
ROI: (192, 192, 144); overlap=0.5; threshold=0.5
✅ Pinned model loaded and checksum verified


In [3]:
# Cell 3 — Reconstruct source inputs and define four controlled perturbations plus one severe challenge

CASE_ORDER = ("stable", "progression", "low-confidence")
MODEL_CHANNELS = (
    ("T1c", ("t1c", "t1ce", "t1gd")),
    ("T1", ("t1",)),
    ("T2", ("t2",)),
    ("FLAIR", ("flair",)),
)

def resolve_modality(modality_files: dict[str, str], aliases: tuple[str, ...]) -> tuple[str, Path]:
    for original_name, relative_path in modality_files.items():
        cleaned = original_name.lower().replace("-", "").replace("_", "")
        for alias in aliases:
            token = alias.lower().replace("-", "").replace("_", "")
            if token in cleaned:
                if token == "t1" and any(x in cleaned for x in ("t1c", "t1ce", "t1gd")):
                    continue
                return original_name, PROJECT_ROOT / relative_path
    raise KeyError(f"Missing modality {aliases} in {sorted(modality_files)}")

imaging_by_id = {c["case_id"]: c for c in imaging_manifest["cases"]}
segmentation_by_id = {c["case_id"]: c for c in segmentation_manifest["cases"]}

cases = []
for case_id in CASE_ORDER:
    img_case = imaging_by_id[case_id]
    seg_case = segmentation_by_id[case_id]
    ordered = []
    for model_channel, aliases in MODEL_CHANNELS:
        source_name, path = resolve_modality(img_case["modality_files"], aliases)
        if not path.exists():
            raise FileNotFoundError(path)
        ordered.append({"model_channel": model_channel, "source_name": source_name, "path": path})

    baseline_mask = PROJECT_ROOT / seg_case["outputs"]["predicted_whole_tumor_mask_file"]
    baseline_prob = PROJECT_ROOT / seg_case["outputs"]["whole_tumor_probability_file"]
    for path in (baseline_mask, baseline_prob):
        if not path.exists() or path.stat().st_size == 0:
            raise FileNotFoundError(path)

    first_image = nib.load(str(ordered[0]["path"]))
    baseline_image = nib.load(str(baseline_mask))
    if first_image.shape != baseline_image.shape:
        raise AssertionError(f"Shape mismatch for {case_id}")
    if not np.allclose(first_image.affine, baseline_image.affine, atol=1e-4, rtol=0):
        raise AssertionError(f"Affine mismatch for {case_id}")

    cases.append({
        "case_id": case_id,
        "source_case_id": seg_case["source_case_id"],
        "patient_reference": seg_case["patient_reference"],
        "followup_imaging_reference": seg_case["followup_imaging_reference"],
        "ordered_modalities": ordered,
        "baseline_mask_path": baseline_mask,
        "baseline_probability_path": baseline_prob,
        "shape": tuple(int(v) for v in first_image.shape),
        "spacing_mm": tuple(float(v) for v in seg_case["spacing_mm"]),
        "planned_future_perturbation": bool(seg_case.get("planned_future_perturbation", False)),
    })

def brain_support(channels: np.ndarray) -> np.ndarray:
    return np.any(np.isfinite(channels) & (channels != 0), axis=0)

def fit_shape(array: np.ndarray, target: tuple[int, int, int]) -> np.ndarray:
    slices = []
    for current, wanted in zip(array.shape, target):
        if current > wanted:
            start = (current - wanted) // 2
            slices.append(slice(start, start + wanted))
        else:
            slices.append(slice(None))
    out = array[tuple(slices)]
    pads = []
    for current, wanted in zip(out.shape, target):
        deficit = max(0, wanted - current)
        pads.append((deficit // 2, deficit - deficit // 2))
    if any(a or b for a, b in pads):
        out = np.pad(out, pads)
    if out.shape != target:
        raise AssertionError(f"Could not restore shape {target}; found {out.shape}")
    return out

def add_noise(channels: np.ndarray, seed: int) -> tuple[np.ndarray, Callable[[np.ndarray], np.ndarray]]:
    rng = np.random.default_rng(seed)
    support = brain_support(channels)
    out = channels.copy().astype(np.float32)
    for index in range(4):
        sd = max(float(out[index][support].std()), 1e-6)
        out[index][support] += rng.normal(0, 0.035 * sd, int(support.sum())).astype(np.float32)
        out[index][~support] = 0
    return out, lambda p: p

def blur(channels: np.ndarray, seed: int) -> tuple[np.ndarray, Callable[[np.ndarray], np.ndarray]]:
    del seed
    support = brain_support(channels)
    out = np.stack([ndimage.gaussian_filter(x, sigma=0.65) for x in channels]).astype(np.float32)
    out[:, ~support] = 0
    return out, lambda p: p

def rotate(channels: np.ndarray, seed: int) -> tuple[np.ndarray, Callable[[np.ndarray], np.ndarray]]:
    del seed
    angle = 2.0
    out = ndimage.rotate(channels, angle, axes=(1, 2), reshape=False, order=1, mode="constant", cval=0, prefilter=False).astype(np.float32)
    def inverse(probabilities: np.ndarray) -> np.ndarray:
        restored = ndimage.rotate(probabilities, -angle, axes=(1, 2), reshape=False, order=1, mode="constant", cval=0, prefilter=False)
        return np.clip(restored, 0, 1).astype(np.float32)
    return out, inverse

def resolution(channels: np.ndarray, seed: int) -> tuple[np.ndarray, Callable[[np.ndarray], np.ndarray]]:
    del seed
    scale = 0.82
    support = brain_support(channels)
    target = channels.shape[1:]
    restored = []
    for channel in channels:
        low = ndimage.zoom(channel, (scale, scale, scale), order=1)
        back = ndimage.zoom(low, tuple(t / c for t, c in zip(target, low.shape)), order=1)
        restored.append(fit_shape(back, target))
    out = np.stack(restored).astype(np.float32)
    out[:, ~support] = 0
    return out, lambda p: p

STANDARD_PERTURBATIONS = {
    "gaussian_noise": add_noise,
    "gaussian_blur": blur,
    "small_rotation": rotate,
    "resolution_degradation": resolution,
}

def severe_challenge(channels: np.ndarray, seed: int) -> tuple[np.ndarray, Callable[[np.ndarray], np.ndarray]]:
    rng = np.random.default_rng(seed)
    support = brain_support(channels)
    out = channels.copy().astype(np.float32)
    for index in (0, 2, 3):
        values = channels[index][support]
        mean = float(values.mean())
        sd = max(float(values.std()), 1e-6)
        noise = rng.normal(mean, sd, support.shape).astype(np.float32)
        noise = ndimage.gaussian_filter(noise, sigma=1.2)
        noise[~support] = 0
        out[index] = noise
    ghost = np.roll(out, shift=10, axis=2)
    out = 0.60 * out + 0.40 * ghost
    out = ndimage.gaussian_filter(out, sigma=(0, 0.8, 0.8, 0.5))
    out[:, ~support] = 0
    return out.astype(np.float32), lambda p: p

definitions = {
    "standard_suite": {
        "gaussian_noise": {"noise_sd_fraction": 0.035},
        "gaussian_blur": {"sigma_voxels": 0.65},
        "small_rotation": {"degrees": 2.0, "inverse_alignment": True},
        "resolution_degradation": {"downsample_scale": 0.82},
    },
    "low_confidence_challenge": {
        "name": "severe_modality_corruption",
        "case_id": "low-confidence",
        "synthetic": True,
        "natural_image_quality_claimed": False,
    },
}
write_json(PERTURBATION_DEFINITIONS_PATH, definitions)

print("=" * 100)
print("✅ Three model-input packages reconstructed")
print(f"✅ Standard perturbations: {list(STANDARD_PERTURBATIONS)}")
print("✅ Separate severe synthetic low-confidence challenge defined")
print("=" * 100)

✅ Three model-input packages reconstructed
✅ Standard perturbations: ['gaussian_noise', 'gaussian_blur', 'small_rotation', 'resolution_degradation']
✅ Separate severe synthetic low-confidence challenge defined


In [4]:
# Cell 4 — Define model inference, stability metrics, plausibility, provenance, and QC scoring

def load_channels(case: dict[str, Any]) -> np.ndarray:
    arrays = []
    for item in case["ordered_modalities"]:
        array = np.asanyarray(nib.load(str(item["path"])).dataobj).astype(np.float32)
        if not np.isfinite(array).all():
            raise ValueError(item["path"])
        arrays.append(array)
    return np.stack(arrays)

def run_inference(channels: np.ndarray) -> tuple[np.ndarray, float]:
    torch.cuda.synchronize()
    started = time.perf_counter()
    probabilities = segmentation_service.infer_probabilities(
        MODEL, channels, DEVICE, ROI_SIZE, overlap=SW_OVERLAP, amp=USE_AMP
    )
    torch.cuda.synchronize()
    return probabilities, time.perf_counter() - started

def wt_mask(probabilities: np.ndarray) -> np.ndarray:
    return segmentation_service.enforce_nested_regions(probabilities, MODEL_THRESHOLD)["whole_tumor"].astype(bool)

def save_mask(mask: np.ndarray, reference_path: Path, destination: Path) -> None:
    reference = nib.load(str(reference_path))
    destination.parent.mkdir(parents=True, exist_ok=True)
    header = reference.header.copy()
    header.set_data_dtype(np.uint8)
    nib.save(nib.Nifti1Image(mask.astype(np.uint8), reference.affine, header), str(destination))

def dice(a: np.ndarray, b: np.ndarray) -> float:
    a, b = a.astype(bool), b.astype(bool)
    denominator = int(a.sum()) + int(b.sum())
    return 1.0 if denominator == 0 else float(2 * np.count_nonzero(a & b) / denominator)

def volume_ml(mask: np.ndarray, spacing: tuple[float, float, float]) -> float:
    return float(np.count_nonzero(mask) * np.prod(spacing) / 1000.0)

def hd95(a: np.ndarray, b: np.ndarray, spacing: tuple[float, float, float]) -> float | None:
    a, b = a.astype(bool), b.astype(bool)
    if not a.any() and not b.any():
        return 0.0
    if not a.any() or not b.any():
        return None
    structure = ndimage.generate_binary_structure(3, 1)
    sa = a ^ ndimage.binary_erosion(a, structure=structure)
    sb = b ^ ndimage.binary_erosion(b, structure=structure)
    if not sa.any(): sa = a
    if not sb.any(): sb = b
    db = ndimage.distance_transform_edt(~sb, sampling=spacing)
    da = ndimage.distance_transform_edt(~sa, sampling=spacing)
    distances = np.concatenate([db[sa], da[sb]])
    return float(np.percentile(distances, 95))

def component_metrics(mask: np.ndarray) -> dict[str, Any]:
    labeled, count = ndimage.label(mask.astype(bool))
    if count == 0:
        return {"component_count": 0, "largest_component_fraction": 0.0, "plausibility_score": 0.0}
    sizes = np.bincount(labeled.ravel())[1:]
    largest = float(sizes.max() / sizes.sum())
    penalty = min(max(count - 1, 0) / 20, 1)
    return {
        "component_count": int(count),
        "largest_component_fraction": round(largest, 6),
        "plausibility_score": round(float(np.clip(0.75 * largest + 0.25 * (1 - penalty), 0, 1)), 6),
    }

def ambiguity(probability: np.ndarray, baseline: np.ndarray) -> dict[str, float]:
    roi = ndimage.binary_dilation(baseline, iterations=5)
    fraction = float((np.abs(probability[roi] - MODEL_THRESHOLD) <= 0.10).mean())
    return {
        "threshold_ambiguity_fraction": round(fraction, 6),
        "threshold_certainty_score": round(float(np.clip(1 - fraction / 0.30, 0, 1)), 6),
    }

def provenance(case: dict[str, Any]) -> dict[str, Any]:
    seg_case = segmentation_by_id[case["case_id"]]
    elements = {
        "source_modality_hashes": bool(seg_case.get("source_modality_sha256")),
        "model_name": bool(seg_case.get("model_bundle_name")),
        "model_version": bool(seg_case.get("model_bundle_version")),
        "model_revision": bool(seg_case.get("model_bundle_revision")),
        "checkpoint_hash": bool(seg_case.get("model_checkpoint_sha256")),
        "inference_timestamp": bool(seg_case.get("generated_utc")),
        "patient_reference": bool(seg_case.get("patient_reference")),
        "imaging_reference": bool(seg_case.get("followup_imaging_reference")),
        "baseline_mask": case["baseline_mask_path"].exists(),
        "baseline_probability": case["baseline_probability_path"].exists(),
    }
    captured = sum(elements.values())
    return {"score": round(captured / len(elements), 6), "captured": captured, "required": len(elements), "elements": elements}

def make_row(case: dict[str, Any], suite: str, name: str, baseline: np.ndarray, result: np.ndarray, seconds: float, output_path: Path) -> dict[str, Any]:
    v0 = volume_ml(baseline, case["spacing_mm"])
    v1 = volume_ml(result, case["spacing_mm"])
    boundary = hd95(baseline, result, case["spacing_mm"])
    return {
        "case_id": case["case_id"],
        "source_case_id": case["source_case_id"],
        "suite": suite,
        "perturbation": name,
        "mask_dice_to_baseline": round(dice(baseline, result), 6),
        "baseline_volume_ml": round(v0, 6),
        "perturbed_volume_ml": round(v1, 6),
        "relative_volume_change": round(0 if v0 <= 0 else abs(v1 - v0) / v0, 6),
        "boundary_hd95_mm": None if boundary is None else round(boundary, 6),
        "perturbed_mask_nonempty": bool(result.any()),
        "inference_seconds": round(seconds, 6),
        "output_mask_file": output_path.relative_to(PROJECT_ROOT).as_posix(),
    }

def qc_score(standard_rows: list[dict[str, Any]], challenge: dict[str, Any] | None, plausibility_score: float, certainty_score: float, provenance_score: float) -> dict[str, Any]:
    min_dice = min(float(r["mask_dice_to_baseline"]) for r in standard_rows)
    max_volume = max(float(r["relative_volume_change"]) for r in standard_rows)
    hd_values = [float(r["boundary_hd95_mm"]) for r in standard_rows if r["boundary_hd95_mm"] is not None]
    max_boundary = max(hd_values) if hd_values else 99.0

    if challenge is not None:
        min_dice = min(min_dice, float(challenge["mask_dice_to_baseline"]))
        max_volume = max(max_volume, float(challenge["relative_volume_change"]))
        max_boundary = max(max_boundary, 99.0 if challenge["boundary_hd95_mm"] is None else float(challenge["boundary_hd95_mm"]))

    agreement = float(np.clip((min_dice - 0.60) / 0.35, 0, 1))
    volume = float(np.clip(1 - max_volume / 0.35, 0, 1))
    boundary = float(np.clip(1 - max_boundary / 15, 0, 1))
    composite = round(
        0.30 * agreement + 0.20 * volume + 0.15 * boundary +
        0.10 * plausibility_score + 0.10 * certainty_score + 0.15 * provenance_score,
        6,
    )

    manual = min_dice < 0.72 or max_volume > 0.30 or max_boundary > 12 or composite < 0.50
    review = min_dice < 0.85 or max_volume > 0.15 or max_boundary > 6 or composite < 0.75
    category = "Manual review required" if manual else "Review recommended" if review else "High confidence"
    return {
        "qc_score": composite,
        "category": category,
        "effective_min_mask_dice": round(min_dice, 6),
        "effective_max_relative_volume_change": round(max_volume, 6),
        "effective_max_boundary_hd95_mm": round(max_boundary, 6),
        "component_scores": {
            "agreement_score": round(agreement, 6),
            "volume_stability_score": round(volume, 6),
            "boundary_stability_score": round(boundary, 6),
            "plausibility_score": round(plausibility_score, 6),
            "threshold_certainty_score": round(certainty_score, 6),
            "provenance_completeness_score": round(provenance_score, 6),
        },
        "thresholds_are_engineering_parameters": True,
        "observation_status_until_review": "preliminary",
        "human_review_required_before_final": True,
        "autonomous_finalization_allowed": False,
    }

print("✅ Robustness metrics, provenance completeness, and QC engine defined")

✅ Robustness metrics, provenance completeness, and QC engine defined


In [5]:
# Cell 5 — Run 12 standard robustness inferences and one severe low-confidence challenge

for folder in (MASK_ROOT, CASE_RESULT_ROOT, PREVIEW_ROOT):
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True, exist_ok=True)

standard_rows = []
runtime_rows = []
cache = {}

for case_index, case in enumerate(cases, start=1):
    case_id = case["case_id"]
    print(f"[{case_index}/3] {case_id}")
    channels = load_channels(case)
    baseline = np.asanyarray(nib.load(str(case["baseline_mask_path"])).dataobj) > 0
    with np.load(case["baseline_probability_path"]) as archive:
        probability = archive["probability"].astype(np.float32)
    cache[case_id] = {"channels": channels, "baseline": baseline, "probability": probability}

    for perturbation_index, (name, function) in enumerate(STANDARD_PERTURBATIONS.items(), start=1):
        perturbed, inverse = function(channels, 5000 + case_index * 100 + perturbation_index)
        probabilities, seconds = run_inference(perturbed)
        result = wt_mask(inverse(probabilities))
        output_path = MASK_ROOT / case_id / f"{name}_whole_tumor_binary.nii.gz"
        save_mask(result, case["baseline_mask_path"], output_path)
        row = make_row(case, "standard", name, baseline, result, seconds, output_path)
        standard_rows.append(row)
        runtime_rows.append({"case_id": case_id, "suite": "standard", "perturbation": name, "seconds": round(seconds, 6)})
        print(f"    {name}: Dice={row['mask_dice_to_baseline']:.4f}; ΔV={100 * row['relative_volume_change']:.1f}%")
        del perturbed, probabilities, result
        torch.cuda.empty_cache()

if len(standard_rows) != 12:
    raise AssertionError(f"Expected 12 standard runs; found {len(standard_rows)}")

low_case = next(c for c in cases if c["case_id"] == "low-confidence")
if not low_case["planned_future_perturbation"]:
    raise AssertionError("Low-confidence case is not marked for the planned challenge")

challenged_channels, inverse = severe_challenge(cache["low-confidence"]["channels"], 9052026)
challenged_probabilities, challenge_seconds = run_inference(challenged_channels)
challenge_mask = wt_mask(inverse(challenged_probabilities))
challenge_output = MASK_ROOT / "low-confidence" / "severe_modality_corruption_whole_tumor_binary.nii.gz"
save_mask(challenge_mask, low_case["baseline_mask_path"], challenge_output)
challenge_row = make_row(
    low_case, "low-confidence-demo-challenge", "severe_modality_corruption",
    cache["low-confidence"]["baseline"], challenge_mask, challenge_seconds, challenge_output,
)
runtime_rows.append({"case_id": "low-confidence", "suite": "challenge", "perturbation": "severe_modality_corruption", "seconds": round(challenge_seconds, 6)})

all_rows = standard_rows + [challenge_row]
pd.DataFrame(all_rows).to_csv(PERTURBATION_RESULTS_CSV, index=False)
write_json(PERTURBATION_RESULTS_JSON, {
    "project_name": project_config["project_name"],
    "generated_utc": utc_now(),
    "standard_run_count": 12,
    "challenge_run_count": 1,
    "challenge_is_synthetic": True,
    "rows": all_rows,
})
write_json(RUNTIME_LOG_PATH, {
    "generated_utc": utc_now(),
    "gpu_name": GPU.name,
    "gpu_memory_gib": round(GPU_MEMORY_GIB, 3),
    "roi_size": list(ROI_SIZE),
    "runs": runtime_rows,
})

print("=" * 100)
print("✅ 12 standard perturbation inferences completed")
print("✅ 1 severe synthetic low-confidence challenge completed")
print(f"Challenge Dice={challenge_row['mask_dice_to_baseline']:.4f}; ΔV={100 * challenge_row['relative_volume_change']:.1f}%")
print("=" * 100)

[1/3] stable
    gaussian_noise: Dice=0.9918; ΔV=1.6%
    gaussian_blur: Dice=0.9907; ΔV=0.6%
    small_rotation: Dice=0.9870; ΔV=2.2%
    resolution_degradation: Dice=0.9910; ΔV=0.8%
[2/3] progression
    gaussian_noise: Dice=0.9993; ΔV=0.0%
    gaussian_blur: Dice=0.9763; ΔV=2.5%
    small_rotation: Dice=0.9821; ΔV=0.0%
    resolution_degradation: Dice=0.9760; ΔV=2.3%
[3/3] low-confidence
    gaussian_noise: Dice=0.9991; ΔV=0.0%
    gaussian_blur: Dice=0.9763; ΔV=2.5%
    small_rotation: Dice=0.9822; ΔV=0.0%
    resolution_degradation: Dice=0.9760; ΔV=2.3%
✅ 12 standard perturbation inferences completed
✅ 1 severe synthetic low-confidence challenge completed
Challenge Dice=0.0000; ΔV=92.1%


In [6]:
# Cell 6 — Assign QC categories, block finalization, and create tables and visual evidence

qc_cases = []
preview_paths = []

for case in cases:
    case_id = case["case_id"]
    baseline = cache[case_id]["baseline"]
    probability = cache[case_id]["probability"]
    case_rows = [r for r in standard_rows if r["case_id"] == case_id]
    challenge = challenge_row if case_id == "low-confidence" else None

    plaus = component_metrics(baseline)
    amb = ambiguity(probability, baseline)
    prov = provenance(case)
    qc = qc_score(case_rows, challenge, plaus["plausibility_score"], amb["threshold_certainty_score"], prov["score"])

    worst = min(case_rows, key=lambda r: float(r["mask_dice_to_baseline"]))
    display_path = challenge_output if case_id == "low-confidence" else PROJECT_ROOT / worst["output_mask_file"]
    display_mask = np.asanyarray(nib.load(str(display_path)).dataobj) > 0
    display_name = "severe_modality_corruption" if case_id == "low-confidence" else worst["perturbation"]

    flair_path = next(x["path"] for x in case["ordered_modalities"] if x["model_channel"] == "FLAIR")
    flair = np.asanyarray(nib.load(str(flair_path)).dataobj).astype(np.float32)
    union = baseline | display_mask
    slice_index = int(np.argmax(union.sum(axis=(0, 1))))
    image = flair[:, :, slice_index]
    values = image[(image != 0) & np.isfinite(image)]
    if values.size:
        lo, hi = np.percentile(values, [1, 99])
        image = np.clip((image - lo) / (max(hi, lo + 1e-6) - lo), 0, 1)
    else:
        image = np.zeros_like(image)

    base_slice = baseline[:, :, slice_index]
    stress_slice = display_mask[:, :, slice_index]
    disagreement = np.logical_xor(base_slice, stress_slice)

    preview = PREVIEW_ROOT / f"{case_id}_robustness_qc.png"
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    axes[0].imshow(np.rot90(image), cmap="gray"); axes[0].set_title(f"{case_id}: FLAIR"); axes[0].axis("off")
    axes[1].imshow(np.rot90(image), cmap="gray"); axes[1].imshow(np.rot90(base_slice.astype(float)), alpha=np.rot90(base_slice.astype(float)) * 0.5); axes[1].set_title("Baseline WT"); axes[1].axis("off")
    axes[2].imshow(np.rot90(image), cmap="gray"); axes[2].imshow(np.rot90(stress_slice.astype(float)), alpha=np.rot90(stress_slice.astype(float)) * 0.5); axes[2].set_title(display_name); axes[2].axis("off")
    axes[3].imshow(np.rot90(image), cmap="gray"); axes[3].imshow(np.rot90(disagreement.astype(float)), alpha=np.rot90(disagreement.astype(float)) * 0.75); axes[3].set_title(f"{qc['category']}\nQC {qc['qc_score']:.3f}"); axes[3].axis("off")
    fig.tight_layout(); fig.savefig(preview, dpi=170, bbox_inches="tight"); plt.close(fig)
    preview_paths.append(preview)

    payload = {
        "case_id": case_id,
        "source_case_id": case["source_case_id"],
        "patient_reference": case["patient_reference"],
        "followup_imaging_reference": case["followup_imaging_reference"],
        "generated_utc": utc_now(),
        "standard_perturbations": case_rows,
        "low_confidence_demo_challenge": challenge,
        "baseline_plausibility": plaus,
        "threshold_ambiguity": amb,
        "provenance_completeness": prov,
        "qc": qc,
        "workflow_state": {
            "ai_result_status": "preliminary",
            "human_review_task_required": True,
            "autonomous_finalization_allowed": False,
            "next_action": "route-for-manual-review" if qc["category"] == "Manual review required" else "present-to-human-reviewer",
        },
        "preview_file": preview.relative_to(PROJECT_ROOT).as_posix(),
    }
    result_path = CASE_RESULT_ROOT / f"{case_id}_qc_result.json"
    write_json(result_path, payload)
    payload["case_result_file"] = result_path.relative_to(PROJECT_ROOT).as_posix()
    qc_cases.append(payload)

low_result = next(c for c in qc_cases if c["case_id"] == "low-confidence")
if low_result["qc"]["category"] != "Manual review required":
    raise AssertionError("Low-confidence challenge was not routed to manual review")
if any(c["workflow_state"]["autonomous_finalization_allowed"] for c in qc_cases):
    raise AssertionError("Autonomous finalization was incorrectly allowed")

write_json(QC_MANIFEST_PATH, {
    "project_name": project_config["project_name"],
    "project_version": project_config.get("version", "0.1.0"),
    "generated_utc": utc_now(),
    "notebook_number": "05",
    "engineering_parameter_notice": "QC weights and thresholds are workflow-design parameters, not clinical safety probabilities.",
    "data_governance": {
        "public_deidentified_imaging_only": True,
        "synthetic_fhir_context_only": True,
        "severe_challenge_is_synthetic": True,
        "clinical_validation_claimed": False,
    },
    "cases": qc_cases,
})

summary_rows = []
for c in qc_cases:
    summary_rows.append({
        "case_id": c["case_id"],
        "source_case_id": c["source_case_id"],
        "qc_score": c["qc"]["qc_score"],
        "qc_category": c["qc"]["category"],
        "effective_min_mask_dice": c["qc"]["effective_min_mask_dice"],
        "effective_max_relative_volume_change": c["qc"]["effective_max_relative_volume_change"],
        "effective_max_boundary_hd95_mm": c["qc"]["effective_max_boundary_hd95_mm"],
        "plausibility_score": c["baseline_plausibility"]["plausibility_score"],
        "threshold_certainty_score": c["threshold_ambiguity"]["threshold_certainty_score"],
        "provenance_completeness_score": c["provenance_completeness"]["score"],
        "observation_status": "preliminary",
        "autonomous_finalization_allowed": False,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(QC_SUMMARY_CSV, index=False)
summary = {
    "project_name": project_config["project_name"],
    "generated_utc": utc_now(),
    "case_count": 3,
    "standard_perturbation_runs": 12,
    "challenge_runs": 1,
    "total_robustness_inference_runs": 13,
    "low_confidence_manual_review_detection_rate": 1.0,
    "preliminary_status_rate": 1.0,
    "autonomous_finalization_block_rate": 1.0,
    "case_preview_count": len(preview_paths),
    "rows": summary_rows,
}
write_json(QC_SUMMARY_JSON, summary)

score_preview = PREVIEW_ROOT / "qc_score_summary.png"
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(summary_df["case_id"], summary_df["qc_score"])
ax.axhline(0.75, linestyle="--", label="High-confidence score gate")
ax.axhline(0.50, linestyle=":", label="Manual-review score gate")
ax.set_ylim(0, 1.05); ax.set_ylabel("Engineering QC score"); ax.set_title("NeuroFHIR-QC workflow triage"); ax.legend()
fig.tight_layout(); fig.savefig(score_preview, dpi=170, bbox_inches="tight"); plt.close(fig)
preview_paths.append(score_preview)

print("=" * 100)
print("✅ QC triage completed")
for row in summary_rows:
    print(f" - {row['case_id']}: {row['qc_category']} | score={row['qc_score']:.3f}")
print("✅ Low-confidence case routed to manual review")
print("✅ All 3 results remain preliminary; autonomous finalization blocked")
print("=" * 100)

✅ QC triage completed
 - stable: High confidence | score=0.963
 - progression: High confidence | score=0.965
 - low-confidence: Manual review required | score=0.339
✅ Low-confidence case routed to manual review
✅ All 3 results remain preliminary; autonomous finalization blocked


In [7]:
# Cell 7 — Create reusable services, verification script, requirements, and documentation

ROBUSTNESS_SERVICE_PATH = PROJECT_ROOT / "backend/app/services/robustness_service.py"
QC_ENGINE_PATH = PROJECT_ROOT / "backend/app/services/qc_engine.py"
VERIFY_SCRIPT_PATH = PROJECT_ROOT / "scripts/verify_robustness_results.py"
REQUIREMENTS_PATH = PROJECT_ROOT / "requirements/robustness.txt"
TECH_DOC_PATH = PROJECT_ROOT / "docs/TRUST_AND_ROBUSTNESS_ENGINE.md"

robustness_source = '''
from __future__ import annotations
import numpy as np
from scipy import ndimage

def dice_coefficient(a: np.ndarray, b: np.ndarray) -> float:
    a, b = a.astype(bool), b.astype(bool)
    denominator = int(a.sum()) + int(b.sum())
    return 1.0 if denominator == 0 else float(2 * np.count_nonzero(a & b) / denominator)

def volume_ml(mask: np.ndarray, spacing_mm: tuple[float, float, float]) -> float:
    return float(np.count_nonzero(mask) * np.prod(spacing_mm) / 1000.0)

def component_plausibility(mask: np.ndarray) -> dict:
    labeled, count = ndimage.label(mask.astype(bool))
    if count == 0:
        return {"component_count": 0, "largest_component_fraction": 0.0, "plausibility_score": 0.0}
    sizes = np.bincount(labeled.ravel())[1:]
    largest = float(sizes.max() / sizes.sum())
    penalty = min(max(count - 1, 0) / 20.0, 1.0)
    return {"component_count": int(count), "largest_component_fraction": largest,
            "plausibility_score": float(np.clip(0.75 * largest + 0.25 * (1 - penalty), 0, 1))}
'''

qc_source = '''
from __future__ import annotations

def clip01(value: float) -> float:
    return max(0.0, min(1.0, float(value)))

def classify_qc(minimum_mask_dice: float, maximum_relative_volume_change: float,
                maximum_boundary_hd95_mm: float, plausibility_score: float,
                threshold_certainty_score: float, provenance_score: float) -> dict:
    agreement = clip01((minimum_mask_dice - 0.60) / 0.35)
    volume = clip01(1.0 - maximum_relative_volume_change / 0.35)
    boundary = clip01(1.0 - maximum_boundary_hd95_mm / 15.0)
    score = (0.30 * agreement + 0.20 * volume + 0.15 * boundary +
             0.10 * clip01(plausibility_score) + 0.10 * clip01(threshold_certainty_score) +
             0.15 * clip01(provenance_score))
    manual = minimum_mask_dice < 0.72 or maximum_relative_volume_change > 0.30 or maximum_boundary_hd95_mm > 12 or score < 0.50
    review = minimum_mask_dice < 0.85 or maximum_relative_volume_change > 0.15 or maximum_boundary_hd95_mm > 6 or score < 0.75
    category = "Manual review required" if manual else "Review recommended" if review else "High confidence"
    return {"qc_score": round(score, 6), "category": category, "observation_status_until_review": "preliminary",
            "autonomous_finalization_allowed": False, "thresholds_are_engineering_parameters": True}
'''

verify_source = '''
from __future__ import annotations
import json
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
MANIFEST = ROOT / "data/sample_biomarkers/notebook_05/qc_case_manifest.json"
SUMMARY = ROOT / "evaluation/results/notebook_05_trust_and_robustness/qc_case_summary.json"

def load(path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def main():
    cases = load(MANIFEST)["cases"]
    summary = load(SUMMARY)
    if len(cases) != 3:
        raise SystemExit("Expected three cases")
    low = next(c for c in cases if c["case_id"] == "low-confidence")
    if low["qc"]["category"] != "Manual review required":
        raise SystemExit("Low-confidence case did not reach manual review")
    if any(c["workflow_state"]["autonomous_finalization_allowed"] for c in cases):
        raise SystemExit("Autonomous finalization was allowed")
    if float(summary.get("preliminary_status_rate", 0)) != 1.0:
        raise SystemExit("Preliminary status failed")
    print("Notebook 05 robustness artifacts passed verification.")

if __name__ == "__main__":
    main()
'''

documentation = f'''
# NeuroFHIR-QC Trust and Robustness Engine

Notebook 05 evaluates four controlled perturbations for each case and a separate severe synthetic challenge for the low-confidence demonstration.

Signals: output Dice agreement, volume stability, boundary HD95, component plausibility, threshold ambiguity, and provenance completeness.

The QC score and thresholds are engineering workflow parameters. They are not clinically validated safety probabilities. Every AI result remains preliminary until explicit human review.

Core outputs:
- `{PERTURBATION_RESULTS_JSON.relative_to(PROJECT_ROOT).as_posix()}`
- `{PERTURBATION_RESULTS_CSV.relative_to(PROJECT_ROOT).as_posix()}`
- `{QC_MANIFEST_PATH.relative_to(PROJECT_ROOT).as_posix()}`
- `{QC_SUMMARY_JSON.relative_to(PROJECT_ROOT).as_posix()}`
- `data/sample_masks/notebook_05/<case>/`
- `evaluation/results/notebook_05_trust_and_robustness/previews/`

Notebook 06 will add longitudinal comparison and temporal-consistency checks.
'''

for path in (ROBUSTNESS_SERVICE_PATH, QC_ENGINE_PATH, VERIFY_SCRIPT_PATH, REQUIREMENTS_PATH, TECH_DOC_PATH):
    path.parent.mkdir(parents=True, exist_ok=True)

ROBUSTNESS_SERVICE_PATH.write_text(textwrap.dedent(robustness_source).strip() + "\n", encoding="utf-8")
QC_ENGINE_PATH.write_text(textwrap.dedent(qc_source).strip() + "\n", encoding="utf-8")
VERIFY_SCRIPT_PATH.write_text(textwrap.dedent(verify_source).strip() + "\n", encoding="utf-8")
REQUIREMENTS_PATH.write_text(
    "\n".join([
        f"monai=={runtime_versions['monai']}",
        f"torch=={runtime_versions['torch'].split('+')[0]}",
        f"nibabel=={runtime_versions['nibabel']}",
        f"numpy=={runtime_versions['numpy']}",
        f"scipy=={runtime_versions['scipy']}",
        f"pandas=={runtime_versions['pandas']}",
        f"matplotlib=={runtime_versions['matplotlib']}",
    ]) + "\n",
    encoding="utf-8",
)
TECH_DOC_PATH.write_text(textwrap.dedent(documentation).strip() + "\n", encoding="utf-8")

for path in (ROBUSTNESS_SERVICE_PATH, QC_ENGINE_PATH, VERIFY_SCRIPT_PATH):
    compile(path.read_text(encoding="utf-8"), str(path), "exec")

subprocess.check_call([sys.executable, str(VERIFY_SCRIPT_PATH)])

print("✅ Reusable robustness service, QC engine, verification script, and documentation created")

✅ Reusable robustness service, QC engine, verification script, and documentation created


In [8]:
# Cell 8 — Final audit, checksums, and notebook-manifest update

qc_manifest = load_json(QC_MANIFEST_PATH)
summary = load_json(QC_SUMMARY_JSON)
perturbations = load_json(PERTURBATION_RESULTS_JSON)

if len(qc_manifest.get("cases", [])) != 3:
    raise AssertionError("Expected three QC cases")
if summary.get("standard_perturbation_runs") != 12:
    raise AssertionError("Expected 12 standard runs")
if summary.get("challenge_runs") != 1:
    raise AssertionError("Expected one challenge run")
if summary.get("total_robustness_inference_runs") != 13:
    raise AssertionError("Expected 13 total inference runs")
if float(summary.get("low_confidence_manual_review_detection_rate", 0)) != 1.0:
    raise AssertionError("Low-confidence detection failed")
if float(summary.get("preliminary_status_rate", 0)) != 1.0:
    raise AssertionError("Preliminary-status gate failed")
if float(summary.get("autonomous_finalization_block_rate", 0)) != 1.0:
    raise AssertionError("Autonomous-finalization block failed")
if len(perturbations.get("rows", [])) != 13:
    raise AssertionError("Expected 13 perturbation rows")

core_files = [
    PERTURBATION_DEFINITIONS_PATH, PERTURBATION_RESULTS_JSON, PERTURBATION_RESULTS_CSV,
    QC_MANIFEST_PATH, QC_SUMMARY_JSON, QC_SUMMARY_CSV, RUNTIME_LOG_PATH,
    ROBUSTNESS_SERVICE_PATH, QC_ENGINE_PATH, VERIFY_SCRIPT_PATH, REQUIREMENTS_PATH, TECH_DOC_PATH,
]
core_files += sorted(MASK_ROOT.glob("*/*.nii.gz"))
core_files += sorted(CASE_RESULT_ROOT.glob("*.json"))
core_files += sorted(PREVIEW_ROOT.glob("*.png"))
missing = [str(p) for p in core_files if not p.exists() or p.stat().st_size == 0]
if missing:
    raise FileNotFoundError("Missing Notebook 05 evidence:\n" + "\n".join(missing))

geometry = []
for case in qc_manifest["cases"]:
    baseline_path = segmentation_by_id[case["case_id"]]["outputs"]["predicted_whole_tumor_mask_file"]
    baseline_image = nib.load(str(PROJECT_ROOT / baseline_path))
    rows = list(case["standard_perturbations"])
    if case["low_confidence_demo_challenge"] is not None:
        rows.append(case["low_confidence_demo_challenge"])
    for row in rows:
        image = nib.load(str(PROJECT_ROOT / row["output_mask_file"]))
        shape_ok = image.shape == baseline_image.shape
        affine_ok = np.allclose(image.affine, baseline_image.affine, atol=1e-4, rtol=0)
        if not (shape_ok and affine_ok):
            raise AssertionError(row["output_mask_file"])
        geometry.append({"case_id": case["case_id"], "file": row["output_mask_file"], "shape_match": shape_ok, "affine_match": affine_ok})

checksums = []
for path in sorted({p.resolve() for p in core_files}, key=str):
    checksums.append({
        "relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })

final_audit = {
    "project_name": project_config["project_name"],
    "project_version": project_config.get("version", "0.1.0"),
    "notebook_number": "05",
    "notebook_filename": NOTEBOOK_FILENAME,
    "status": "completed",
    "audited_utc": utc_now(),
    "notebook_saved_in_drive": NOTEBOOK_SAVE_PATH.exists() and NOTEBOOK_SAVE_PATH.stat().st_size > 0,
    "metrics": {
        "case_count": 3,
        "standard_perturbation_type_count": 4,
        "standard_perturbation_run_count": 12,
        "challenge_run_count": 1,
        "total_robustness_inference_run_count": 13,
        "low_confidence_manual_review_detection_rate": 1.0,
        "preliminary_status_rate": 1.0,
        "autonomous_finalization_block_rate": 1.0,
        "preview_count": len(list(PREVIEW_ROOT.glob("*.png"))),
    },
    "geometry_validation": geometry,
    "scope": {
        "controlled_perturbations_run": True,
        "robustness_metrics_calculated": True,
        "qc_classification_calculated": True,
        "low_confidence_failure_detected": True,
        "autonomous_finalization_blocked": True,
        "longitudinal_analysis_calculated": False,
        "current_ai_observation_created": False,
        "human_review_transition_executed": False,
        "ai_result_writeback_performed": False,
    },
    "safety": {
        "all_ai_outputs_preliminary": True,
        "human_review_required_before_final": True,
        "thresholds_are_engineering_parameters": True,
        "severe_challenge_is_synthetic": True,
        "clinical_validation_claimed": False,
    },
    "checksum_inventory": checksums,
    "next_notebook": "06 — Longitudinal Analysis",
}
write_json(AUDIT_JSON_PATH, final_audit)

outcomes = "\n".join(f"- {r['case_id']}: {r['qc_category']} (score {r['qc_score']:.3f})" for r in summary["rows"])
AUDIT_MD_PATH.write_text(textwrap.dedent(f"""
# Notebook 05 — Trust and Robustness Engine

**Status:** completed
**Audited:** {final_audit['audited_utc']}

- 12 standard perturbation inference runs completed.
- One separate severe synthetic low-confidence challenge completed.
- Low-confidence manual-review detection: 100%.
- Preliminary-status retention: 100%.
- Autonomous-finalization blocking: 100%.

## QC outcomes
{outcomes}

The score thresholds are engineering workflow parameters, not clinical validation. Notebook 05 did not calculate longitudinal change, execute human review, create a current FHIR AI Observation, or perform write-back.
""").strip() + "\n", encoding="utf-8")

nb05_entry["status"] = "completed"
nb05_entry["completed_utc"] = final_audit["audited_utc"]
nb05_entry["standard_perturbation_run_count"] = 12
nb05_entry["total_robustness_inference_run_count"] = 13
nb05_entry["low_confidence_manual_review_detection_rate"] = 1.0
nb05_entry["autonomous_finalization_block_rate"] = 1.0
nb05_entry["audit_path"] = AUDIT_JSON_PATH.relative_to(PROJECT_ROOT).as_posix()
write_json(NOTEBOOK_MANIFEST_PATH, notebook_manifest)

print("=" * 100)
print("✅ Notebook 05 trust and robustness evidence passed")
print("✅ 12 standard perturbation runs + 1 severe challenge archived")
print("✅ Low-confidence case routed to manual review")
print("✅ All AI results remain preliminary; autonomous finalization blocked 3/3")
print(f"✅ Audit JSON: {AUDIT_JSON_PATH}")
print(f"📓 Manifest status: completed")
print("🎯 Notebook 05 is complete; copy the entire executed notebook to GitHub")
print("➡️ Notebook 06 — Longitudinal Analysis may begin after the GitHub copy preserves these outputs")
print("=" * 100)

✅ Notebook 05 trust and robustness evidence passed
✅ 12 standard perturbation runs + 1 severe challenge archived
✅ Low-confidence case routed to manual review
✅ All AI results remain preliminary; autonomous finalization blocked 3/3
✅ Audit JSON: /content/drive/MyDrive/neurofhir-qc/evaluation/results/notebook_05_trust_robustness_audit.json
📓 Manifest status: completed
🎯 Notebook 05 is complete; copy the entire executed notebook to GitHub
➡️ Notebook 06 — Longitudinal Analysis may begin after the GitHub copy preserves these outputs
